Kết nối drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Cell 1: Làm sạch cho file dữ liệu itviec_jobs.csv

In [ ]:
import pandas as pd
import numpy as np
import re
import unicodedata
from pathlib import Path

print(" LÀM SẠCH & FEATURE ENGINEERING TẬP ITVIEC")

# 1. ĐỌC FILE & XÓA DUPLICATE

input_path = Path('/content/drive/MyDrive/ProjectBigData/itviec_jobs.csv')
df_itviec = pd.read_csv(input_path, on_bad_lines='skip')
before_rows = len(df_itviec)
duplicate_check_columns = [col for col in df_itviec.columns if col not in ['url', 'crawl_time']]
df_itviec = df_itviec.drop_duplicates(subset=duplicate_check_columns).copy()
after_rows = len(df_itviec)
print(f"[1] Duplicate: {before_rows} → {after_rows} dòng (loại {before_rows - after_rows})")
# Chuẩn hóa Unicode NFC toàn bộ cột text
for col in df_itviec.select_dtypes(include=['object']).columns:
    df_itviec[col] = df_itviec[col].apply(
        lambda x: unicodedata.normalize('NFC', str(x)) if pd.notna(x) else x
    )

# 2. CHUẨN HÓA TITLE & JOB LEVEL

def clean_title(t):
    if pd.isna(t): return ""
    t = str(t).lower().strip()
    t = re.sub(r'\bsr\.?\b', 'senior', t)
    t = re.sub(r'\bjr\.?\b', 'junior', t)
    t = re.sub(r'\bdev\b', 'developer', t)  # word boundary tránh "DeveloperOps"
    return t.title()
df_itviec['title_clean'] = df_itviec['title'].apply(clean_title)
def extract_level(title):
    t = str(title).lower()
    if any(x in t for x in ['manager', 'giám đốc', 'head']): return 'Manager'
    if 'lead' in t: return 'Lead'
    if any(x in t for x in ['senior', 'cao cấp', 'chuyên gia', 'expert', 'principal']): return 'Senior'
    if any(x in t for x in ['junior', 'chuyên viên', 'staff', 'nhân viên']): return 'Junior'
    if any(x in t for x in ['fresher', 'intern', 'thực tập']): return 'Fresher'
    return 'Undefined'
df_itviec['job_level'] = df_itviec['title_clean'].apply(extract_level)
print(f"[2] Job level: {df_itviec['job_level'].value_counts().to_dict()}")

# 3. TRÍCH XUẤT YOE (Có Fallback)

def extract_yoe(desc_str):
    if pd.isna(desc_str): return np.nan
    desc = str(desc_str).lower()
    pattern = r'(\d+)\s*(?:\+|-|~|to|đến)?\s*(\d+)?\s*(?:năm|years?|yrs?|yoe)'
    matches = re.findall(pattern, desc)
    if matches:
        try:
            first_match = matches[0]
            min_year = float(first_match[0])
            if first_match[1] != '':
                return (min_year + float(first_match[1])) / 2
            else:
                return min_year if min_year <= 15 else np.nan
        except:
            pass
    fallback_patterns = [
        r'at least\s+(\d+)', r'minimum\s+(\d+)', r'over\s+(\d+)', r'more than\s+(\d+)',
        r'tối thiểu\s+(\d+)', r'trên\s+(\d+)'
    ]
    for fb in fallback_patterns:
        match = re.search(fb, desc)
        if match:
            val = float(match.group(1))
            if val <= 15: return val
    return np.nan
df_itviec['yoe_extracted'] = df_itviec['description'].apply(extract_yoe)

# 4. XỬ LÝ LƯƠNG 

# --- 4.1. TẦNG 1: Làm sạch text thô  ---
def normalize_salary_unit(text: str) -> str:
    """Xử lý lỗi 'VND USD' cùng ô hoặc 'USD USD' lặp lại."""
    text = text.strip()
    # Có cả VND lẫn USD → xóa USD (ưu tiên VND)
    if 'VND' in text and 'USD' in text:
        text = re.sub(r'\bUSD\b', '', text)
        return re.sub(r'\s+', ' ', text).strip()
    # USD xuất hiện nhiều hơn 1 lần → chỉ giữ lần đầu
    usd_matches = list(re.finditer(r'\bUSD\b', text))
    if len(usd_matches) > 1:
        first_end = usd_matches[0].end()
        result = text[:first_end]
        last_idx = first_end
        for m in usd_matches[1:]:
            result += text[last_idx:m.start()]
            last_idx = m.end()
        result += text[last_idx:]
        return re.sub(r'\s+', ' ', result).strip()
    return text
def normalize_special_phrase(text: str) -> str:
    """Chuẩn hóa các cụm đặc biệt bị encode lỗi."""
    lower = text.lower().strip()
    if re.fullmatch(r"-?pen[- ]?neg[- ]?ia[- ]?i[- ]?n(?:\s+usd)?", lower):
        return 'Open to negotiation'
    if re.fullmatch(r"le[- ]*'?s[- ]?discuss(?:\s+usd)?", lower):
        return "Let's discuss"
    if re.fullmatch(r"neg[- ]?ia[- ]?i[- ]?n(?:\s+usd)?", lower):
        return 'Negotiable'
    # Các dạng bị mã hóa lạ của "You'll love it in USD"
    if re.search(r"y[- ]*['\-]?u[- ]*['\-]?ll", lower) and 'ove' in lower:
        return "You'll love it"
    return ''
NEGOTIATION_LABELS = {
    "You'll love it", "Attractive", "Competitive",
    "Negotiable", "Let's discuss", "Open to negotiation",
    "Negotiation", "Thương lượng"
}
def normalize_text_salary(text: str) -> str:
    """Chuẩn hóa text: dấu câu, viết tắt đơn vị, etc."""
    text = text.strip()
    # Dấu gạch ngang đa dạng → chuẩn hóa về '-'
    text = text.replace('−', '-').replace('–', '-').replace('—', '-')
    text = re.sub(r"['']", "'", text)
    text = re.sub(r'[""]', '"', text)
    text = re.sub(r'\s*-\s*', '-', text)
    # Các từ viết tắt đơn vị
    text = re.sub(r'(?i)\bup[- ]*to\b', 'Up to', text)
    text = re.sub(r'(?i)\bgr[- ]?ss\b', '', text)           # gross → xóa
    text = re.sub(r'(?i)\bne[- ]?usd\b', 'USD', text)       # ne-USD → USD
    text = re.sub(r'(?i)\bmil\b', 'm', text)
    text = re.sub(r'(?i)\btri[eệ]u\b', 'm', text)
    text = re.sub(r'(?i)\btr\b', 'm', text)
    text = re.sub(r'(?<=\d)-?r\b', 'm', text)               # 20-r → 20m
    text = re.sub(r'\s+', ' ', text).strip()
    # Kiểm tra special phrase
    special = normalize_special_phrase(text)
    if special:
        return special
    return text
# --- 4.2. TẦNG 2: Trích xuất số (từ Code 2) ---
def normalize_number_token(token: str):
    """Parse số, xử lý dấu chấm/phẩy phân cách hàng nghìn hoặc thập phân."""
    token = re.sub(r'[^\d.,]', '', token.replace(' ', ''))
    if not token:
        return None
    if not re.search(r'[.,]', token):
        return float(token)
    parts = [p for p in re.split(r'[.,]', token) if p != '']
    # Dạng 1.000.000 hoặc 1,000,000 → tất cả phần sau dấu đầu có 3 chữ số
    if len(parts) > 1 and all(len(p) == 3 for p in parts[1:]):
        return float(''.join(parts))
    # Dạng 1.5 hoặc 1,5 → số thập phân
    if len(parts) == 2:
        left, right = parts
        if len(right) in (1, 2):
            return float(f'{left}.{right}')
        return float(left + right)
    return float(''.join(parts))
def extract_numbers_from_salary(text: str):
    candidates = re.findall(r'\d+(?:[.,]\d+)?', text)
    nums = []
    for c in candidates:
        v = normalize_number_token(c)
        if v is not None:
            nums.append(v)
    return nums
# --- 4.3. TẦNG 3: Suy luận tiền tệ  ---
def infer_currency(text: str, numbers: list) -> str:
    """Suy luận tiền tệ dựa trên từ khóa và magnitude của số."""
    upper = text.upper()
    # Tường minh
    if any(kw in upper for kw in ['VND', 'VNĐ', 'DONG', '₫']) or 'Đ' in text:
        return 'VND'
    if '$' in text and 'USD' not in upper:
        return 'USD'
    if 'USD' in upper:
        # Số < 100 → có thể là triệu VND viết sai, nhưng USD thì không hợp lý
        if any(n > 100_000 for n in numbers):
            return 'VND'
        if any(n < 100 for n in numbers):
            return 'VND'
        # Khoảng 500–9999 → hợp lệ USD
        if any(500 <= n <= 9_999 for n in numbers):
            return 'USD'
        return 'USD'
    # Không có từ khóa → suy từ magnitude
    if any(n > 100_000 for n in numbers):
        return 'VND'
    if any(n < 100 for n in numbers):
        return 'VND'        # vd: "10 triệu" bị mất đơn vị → coi là VND
    if any(500 <= n <= 9_999 for n in numbers):
        return 'USD'
    return 'VND'            # mặc định VND cho thị trường VN
def format_m(value: float) -> str:
    """Format số triệu: 12m, 12.5m, etc."""
    if abs(value - round(value)) < 1e-9:
        return f'{int(round(value))}m'
    return f'{value:.2f}m'.rstrip('0').rstrip('.')
# --- 4.4. TẦNG 4: Main salary cleaner → trả về (salary_clean, min, max, avg) ---
USD_TO_VND = 25_400
ANNUAL_MONTHS = 12
MIN_VALID_VND = 1_000_000
MAX_VALID_VND = 200_000_000
def process_salary(raw_value) -> dict:
    """
    Làm sạch và parse salary.
    Trả về dict:
        salary_clean  : str  (vd "12m-20m VND", "1500-2000 USD", "Negotiable", "")
        salary_min_vnd: float hoặc NaN
        salary_max_vnd: float hoặc NaN
        salary_avg_vnd: float hoặc NaN
    """
    nan_result = dict(salary_clean='', salary_min_vnd=np.nan,
                      salary_max_vnd=np.nan, salary_avg_vnd=np.nan)
    if pd.isna(raw_value):
        return nan_result
    text = str(raw_value).strip()
    if not text or text.lower() in {'nan', 'none', ''}:
        return nan_result
    # --- Bước 1: normalize unit (lỗi VND+USD lặp) ---
    text = normalize_salary_unit(text)
    # --- Bước 2: normalize text (viết tắt, dấu câu) ---
    text = normalize_text_salary(text)
    # --- Bước 3: Kiểm tra negotiation ---
    if text in NEGOTIATION_LABELS:
        return dict(salary_clean=text, salary_min_vnd=np.nan,
                    salary_max_vnd=np.nan, salary_avg_vnd=np.nan)
    # --- Bước 4: Xử lý đơn vị K (1.5k → 1500) ---
    text_for_parse = re.sub(
        r'(\d+(?:\.\d+)?)\s*[kK]\b',
        lambda m: str(float(m.group(1)) * 1000),
        text
    )
    # --- Bước 5: Trích xuất số ---
    numbers = extract_numbers_from_salary(text_for_parse)
    if not numbers:
        return nan_result
    # --- Bước 6: Suy luận tiền tệ & annual ---
    currency = infer_currency(text, numbers)
    is_annual = bool(re.search(r'\bnăm\b|\byear\b|\bannual\b', text.lower()))
    is_upto = text.lower().startswith('up')
    # --- Bước 7: Xác định min/max từ danh sách số ---
    min_sal, max_sal = np.nan, np.nan
    if len(numbers) >= 2:
        min_sal, max_sal = numbers[0], numbers[1]
    elif len(numbers) == 1:
        if is_upto or any(kw in text.lower() for kw in ['tới', 'upto', 'lên đến', 'max', 'up to']):
            max_sal = numbers[0]
        elif any(kw in text.lower() for kw in ['từ', 'trên', 'min', '>']):
            min_sal = numbers[0]
        else:
            min_sal = max_sal = numbers[0]
    # --- Bước 8: Quy đổi về VND ---
    def to_vnd(n, currency, is_annual):
        if pd.isna(n): return np.nan
        if currency == 'USD': n *= USD_TO_VND
        if is_annual: n /= ANNUAL_MONTHS
        return n
    min_vnd = to_vnd(min_sal, currency, is_annual)
    max_vnd = to_vnd(max_sal, currency, is_annual)
    # --- Bước 9: Kiểm tra khoảng hợp lệ (Code 1) ---
    avg_vnd = float(np.nanmean([v for v in [min_vnd, max_vnd] if not np.isnan(v)] or [np.nan]))
    if np.isnan(avg_vnd) or avg_vnd < MIN_VALID_VND or avg_vnd > MAX_VALID_VND:
        return nan_result
    # --- Bước 10: Tạo salary_clean string đẹp (Code 2) ---
    if currency == 'VND':
        parts = []
        for n in numbers[:2]:
            v = n / 1_000_000 if n > 100_000 else n
            parts.append(format_m(v))
        salary_clean = '-'.join(parts) + ' VND'
    else:
        parts = []
        for n in numbers[:2]:
            parts.append(str(int(round(n))) if abs(n - round(n)) < 1e-9 else str(n))
        salary_clean = '-'.join(parts) + ' USD'
    if is_upto:
        salary_clean = 'Up to ' + salary_clean
    return dict(
        salary_clean=salary_clean,
        salary_min_vnd=min_vnd,
        salary_max_vnd=max_vnd,
        salary_avg_vnd=avg_vnd
    )
# Áp dụng và bung thành các cột riêng 
print("[4] Đang parse salary...")
salary_results = df_itviec['salary'].apply(process_salary).tolist()
df_sal_parsed = pd.DataFrame(salary_results)
df_itviec = pd.concat([df_itviec.reset_index(drop=True), df_sal_parsed], axis=1)
sal_parsed_count = df_sal_parsed['salary_avg_vnd'].notna().sum()
sal_negotiation_count = df_sal_parsed['salary_clean'].isin(NEGOTIATION_LABELS).sum()
print(f"    → Có số liệu VND: {sal_parsed_count} dòng")
print(f"    → Thương lượng:   {sal_negotiation_count} dòng")
print(f"    → Trống/lỗi:      {len(df_itviec) - sal_parsed_count - sal_negotiation_count} dòng")

# 5. CHUẨN HÓA LOCATION & IS_REMOTE

def standardize_location(loc_str):
    if pd.isna(loc_str): return 'Khác'
    s = str(loc_str).lower()

    locations = []

    # Dùng \b để khớp chính xác từ đứng độc lập, an toàn với các từ viết tắt HN, DN
    if re.search(r'\b(hồ chí minh|hcm|ho chi minh|saigon)\b', s):
        locations.append('Hồ Chí Minh')

    if re.search(r'\b(hà nội|ha noi|hn)\b', s):
        locations.append('Hà Nội')

    if re.search(r'\b(đà nẵng|da nang|dn)\b', s):
        locations.append('Đà Nẵng')

    if 'remote' in s or 'từ xa' in s:
        locations.append('Remote')

    # Nối các địa điểm tìm được thành chuỗi cách nhau bằng dấu phẩy
    if len(locations) > 0:
        return ', '.join(locations)

    return 'Khác'

df_itviec['location_clean'] = df_itviec['location'].apply(standardize_location)

# Dùng .str.contains(). Để bắt được những dòng đa trị như "Hồ Chí Minh, Remote"
df_itviec['is_remote'] = df_itviec['location_clean'].str.contains('Remote', na=False).astype(int)
print(f"[5] Location: {df_itviec['location_clean'].value_counts().to_dict()}")

# 6. LÀM SẠCH DESCRIPTION & SKILLS

def clean_description(text):
    if pd.isna(text): return ""
    text = str(text)
    text = re.sub(r'<[^>]+>', ' ', text)
    for pattern in [
        r'(?i)Your Skills and Experience', r"(?i)Why You'll Love Working Here",
        r'(?i)Responsibilities', r'(?i)Requirements', r'(?i)Job Description',
        r'(?i)Benefits', r'(?i)About our client', r'(?i)About the company'
    ]:
        text = re.sub(pattern, '', text)
    text = re.sub(r'http\S+|www\.\S+|\S+@\S+', '', text)
    text = re.sub(r'[\r\n\t\|\*•]', ' ', text)
    return re.sub(r'\s+', ' ', text).strip()
df_itviec['description_clean'] = df_itviec['description'].apply(clean_description)
def clean_skills(skills_str):
    if pd.isna(skills_str): return ''
    skills = [s.strip().lower() for s in str(skills_str).split(',') if s.strip()]
    skills = list(dict.fromkeys(skills))   # giữ thứ tự, loại duplicate
    return ', '.join(skills)
df_itviec['skills_clean'] = df_itviec['skills'].apply(clean_skills)
df_itviec['skill_count'] = df_itviec['skills_clean'].apply(
    lambda x: len(x.split(',')) if x else 0
)

# 7. XUẤT FILE

df_itviec['source'] = 'ITviec'
final_cols = [
    'source', 'title_clean', 'company',
    'salary',           # raw salary gốc (sau khi normalize)
    'salary_clean',     # salary đã format đẹp (vd "12m-20m VND")
    'salary_min_vnd', 'salary_max_vnd', 'salary_avg_vnd',
    'location_clean', 'is_remote',
    'job_level', 'yoe_extracted',
    'skills_clean', 'skill_count',
    'description_clean', 'url'
]
# Chỉ giữ cột tồn tại trong DataFrame
final_cols = [c for c in final_cols if c in df_itviec.columns]
df_itviec_final = df_itviec[final_cols]
output_path = 'ITVIEC_CLEANED_READY_FINAL.csv'
df_itviec_final.to_csv(output_path, index=False, encoding='utf-8-sig')
print(f" File lưu tại: {output_path}")
print(f"       Tổng dòng: {len(df_itviec_final)}")
print(f"       Cột: {list(df_itviec_final.columns)}")


 LÀM SẠCH & FEATURE ENGINEERING TẬP ITVIEC
[1] Duplicate: 1246 → 762 dòng (loại 484)
[2] Job level: {'Undefined': 403, 'Senior': 217, 'Manager': 63, 'Lead': 52, 'Junior': 19, 'Fresher': 8}
[4] Đang parse salary...
    → Có số liệu VND: 170 dòng
    → Thương lượng:   25 dòng
    → Trống/lỗi:      567 dòng
[5] Location: {'Hồ Chí Minh': 408, 'Hà Nội': 281, 'Hồ Chí Minh, Hà Nội': 31, 'Đà Nẵng': 16, 'Hồ Chí Minh, Đà Nẵng': 16, 'Hồ Chí Minh, Hà Nội, Remote': 3, 'Khác': 2, 'Hồ Chí Minh, Remote': 2, 'Hà Nội, Đà Nẵng, Remote': 1, 'Hà Nội, Đà Nẵng': 1, 'Hồ Chí Minh, Hà Nội, Đà Nẵng, Remote': 1}

[XONG] File lưu tại: ITVIEC_CLEANED_READY_FINAL.csv
       Tổng dòng: 762
       Cột: ['source', 'title_clean', 'company', 'salary', 'salary_clean', 'salary_min_vnd', 'salary_max_vnd', 'salary_avg_vnd', 'location_clean', 'is_remote', 'job_level', 'yoe_extracted', 'skills_clean', 'skill_count', 'description_clean', 'url']


Cell 2: Làm sạch dữ liệu cho file hybrid_jobs_fixed.csv


In [ ]:
import pandas as pd
import numpy as np
import re
import unicodedata
from pathlib import Path

print(" PHASE 1 (VietnamWorks): LÀM SẠCH & CHUẨN HÓA SCHEMA")

# 1. ĐỌC FILE & AUTO-DETECT HEADER + MAP CỘT

input_path = Path('/content/drive/MyDrive/ProjectBigData/hybrid_jobs_fixed.csv')

# --- 1.1. Thử từng header row (0, 1, 2) cho đến khi có cột đọc được ---
def _looks_like_header(cols):
    """Trả về True nếu tên cột trông giống header thật (có chữ, không phải column1...)."""
    generic = sum(1 for c in cols if re.match(r'^(unnamed|column\d+|col\d+|\d+)$', str(c).strip(), re.I))
    return generic < len(cols) // 2  # hơn nửa cột có tên thật → OK

df_vnw = None
for _skip in [0, 1, 2, 3]:
    _df = pd.read_csv(input_path, header=_skip, on_bad_lines='skip')
    _df.columns = [str(c).strip().lower() for c in _df.columns]
    if _looks_like_header(list(_df.columns)):
        df_vnw = _df
        print(f"[1] Header tìm thấy ở dòng {_skip}")
        break

if df_vnw is None:
    # Fallback: đọc không có header, rồi map thủ công theo vị trí
    df_vnw = pd.read_csv(input_path, header=None, on_bad_lines='skip')
    print(f"[1] Không tìm thấy header → đọc theo vị trí cột")

# --- 1.2. Map tên cột nếu cột vẫn là generic (column1, column2...) ---
EXPECTED_COLS = ['title', 'company', 'salary', 'location', 'description', 'skills', 'url', 'crawl_time']

def _map_columns_by_content(df, expected):
    """
    Với mỗi cột trong expected, tìm cột nào trong df có nội dung khớp nhất.
    Ưu tiên: tên cột khớp trực tiếp → keyword trong tên → nội dung mẫu.
    """
    current_cols = list(df.columns)
    # Nếu đã có đủ tên đúng → không cần map
    matched = [c for c in expected if c in current_cols]
    if len(matched) >= 4:
        return df  # đủ cột quan trọng

    mapping = {}
    used = set()

    # Bước 1: khớp trực tiếp (tên giống nhau)
    for exp in expected:
        for col in current_cols:
            if col == exp and col not in used:
                mapping[col] = exp
                used.add(col)
                break

    # Bước 2: khớp theo keyword trong tên cột
    keyword_hints = {
        'title':       ['title', 'job', 'tên', 'chức danh', 'vị trí'],
        'company':     ['company', 'employer', 'công ty', 'doanh nghiệp'],
        'salary':      ['salary', 'wage', 'lương', 'thu nhập', 'income'],
        'location':    ['location', 'city', 'địa điểm', 'tỉnh', 'thành phố'],
        'description': ['description', 'desc', 'content', 'mô tả', 'nội dung'],
        'skills':      ['skill', 'tag', 'kỹ năng', 'tech'],
        'url':         ['url', 'link', 'href'],
        'crawl_time':  ['crawl', 'time', 'date', 'ngày'],
    }
    for exp, hints in keyword_hints.items():
        if exp in mapping.values():
            continue
        for col in current_cols:
            if col in used:
                continue
            if any(h in str(col) for h in hints):
                mapping[col] = exp
                used.add(col)
                break

    # Bước 3: với cột generic còn lại → suy từ nội dung mẫu
    content_hints = {
        'url':         lambda s: bool(re.search(r'https?://', str(s))),
        'salary':      lambda s: bool(re.search(r'\d.*(?:usd|vnd|triệu|tr\b|\$)', str(s), re.I)),
        'description': lambda s: len(str(s)) > 200,
        'skills':      lambda s: ',' in str(s) and len(str(s)) < 300,
        'location':    lambda s: any(k in str(s).lower() for k in ['hà nội', 'hồ chí minh', 'hcm']),
    }
    remaining_generic = [c for c in current_cols if c not in used and
                         re.match(r'^(unnamed|column\d+|col\d+|\d+)$', str(c), re.I)]
    sample = df.dropna().head(20)
    for col in remaining_generic:
        for exp, test_fn in content_hints.items():
            if exp in mapping.values():
                continue
            try:
                hits = sample[col].apply(test_fn).sum()
                if hits >= 3:
                    mapping[col] = exp
                    used.add(col)
                    break
            except Exception:
                pass

    if mapping:
        df = df.rename(columns=mapping)
        print(f"    → Đã map cột: {mapping}")
    return df

df_vnw = _map_columns_by_content(df_vnw, EXPECTED_COLS)

# --- 1.3. Kiểm tra cột bắt buộc ---
required_cols = ['title', 'company', 'salary', 'location']
missing = [c for c in required_cols if c not in df_vnw.columns]
if missing:
    print(f"    [CẢNH BÁO] Vẫn thiếu cột: {missing}")
    print(f"    Cột hiện tại: {list(df_vnw.columns)}")
    print("    → Vui lòng kiểm tra file CSV và điều chỉnh mapping thủ công.")
    # Tạo cột rỗng để pipeline không crash
    for c in missing:
        df_vnw[c] = np.nan

print(f"[1] Đọc file: {len(df_vnw)} dòng | Cột: {list(df_vnw.columns)}")

# 2. XÓA TRÙNG LẶP & CHUẨN HÓA UNICODE

initial_rows = len(df_vnw)
# Dùng tất cả cột quan trọng, bỏ url/crawl_time (thay đổi theo crawl)
duplicate_check_columns = [col for col in df_vnw.columns if col not in ['url', 'crawl_time']]
df_vnw = df_vnw.drop_duplicates(subset=duplicate_check_columns).copy()

for col in df_vnw.select_dtypes(include=['object']).columns:
    df_vnw[col] = df_vnw[col].apply(
        lambda x: unicodedata.normalize('NFC', str(x)) if pd.notna(x) else x
    )
print(f"[2] Duplicate: {initial_rows} → {len(df_vnw)} dòng (loại {initial_rows - len(df_vnw)})")

# 3. CHUẨN HÓA TITLE & JOB LEVEL
def clean_title(t):
    if pd.isna(t): return ""
    t = str(t).strip()
    # Xóa nội dung trong ngoặc đơn (vd: "Dev (Remote)", "(Urgent)")
    t = re.sub(r'\(.*?\)', '', t)
    # Xóa hậu tố mã số đặc thù VietnamWorks (vd: "Dev - Mã số: VNW123")
    t = re.sub(r'\s*[-–]\s*mã\s*số.*$', '', t, flags=re.IGNORECASE)
    t = t.lower().strip()
    t = re.sub(r'\bsr\.?\b', 'senior', t)
    t = re.sub(r'\bjr\.?\b', 'junior', t)
    t = re.sub(r'\bdev\b', 'developer', t)  # word boundary tránh "DeveloperOps"
    return re.sub(r'\s+', ' ', t).strip().title()

df_vnw['title_clean'] = df_vnw['title'].apply(clean_title)

def extract_level(title):
    t = str(title).lower()
    if any(x in t for x in ['manager', 'giám đốc', 'head']): return 'Manager'
    if 'lead' in t: return 'Lead'
    if any(x in t for x in ['senior', 'cao cấp', 'chuyên gia', 'expert', 'principal']): return 'Senior'
    if any(x in t for x in ['junior', 'chuyên viên', 'staff', 'nhân viên']): return 'Junior'
    if any(x in t for x in ['fresher', 'intern', 'thực tập']): return 'Fresher'
    return 'Undefined'

df_vnw['job_level'] = df_vnw['title_clean'].apply(extract_level)
print(f"[3] Job level: {df_vnw['job_level'].value_counts().to_dict()}")

# 4. TRÍCH XUẤT YOE
def extract_yoe(desc_str):
    if pd.isna(desc_str): return np.nan
    desc = str(desc_str).lower()
    pattern = r'(\d+)\s*(?:\+|-|~|to|đến)?\s*(\d+)?\s*(?:năm|years?|yrs?|yoe)'
    matches = re.findall(pattern, desc)
    if matches:
        try:
            first_match = matches[0]
            min_year = float(first_match[0])
            if first_match[1] != '':
                return (min_year + float(first_match[1])) / 2
            else:
                return min_year if min_year <= 15 else np.nan
        except:
            pass
    fallback_patterns = [
        r'at least\s+(\d+)', r'minimum\s+(\d+)',
        r'over\s+(\d+)', r'more than\s+(\d+)',
        r'tối thiểu\s+(\d+)', r'trên\s+(\d+)'
    ]
    for fb in fallback_patterns:
        match = re.search(fb, desc)
        if match:
            val = float(match.group(1))
            if val <= 15: return val
    return np.nan

df_vnw['yoe_extracted'] = df_vnw['description'].apply(extract_yoe)

# 5. XỬ LÝ LƯƠNG
# --- 5.1. Làm sạch text thô ---

def normalize_salary_unit(text: str) -> str:
    """Sửa lỗi 'VND USD' cùng ô, hoặc 'USD USD' lặp lại."""
    text = text.strip()
    if 'VND' in text and 'USD' in text:
        text = re.sub(r'\bUSD\b', '', text)
        return re.sub(r'\s+', ' ', text).strip()
    usd_matches = list(re.finditer(r'\bUSD\b', text))
    if len(usd_matches) > 1:
        first_end = usd_matches[0].end()
        result = text[:first_end]
        last_idx = first_end
        for m in usd_matches[1:]:
            result += text[last_idx:m.start()]
            last_idx = m.end()
        result += text[last_idx:]
        return re.sub(r'\s+', ' ', result).strip()
    return text


def normalize_special_phrase(text: str) -> str:
    """Chuẩn hóa cụm từ bị encode lỗi."""
    lower = text.lower().strip()
    if re.fullmatch(r"-?pen[- ]?neg[- ]?ia[- ]?i[- ]?n(?:\s+usd)?", lower):
        return 'Open to negotiation'
    if re.fullmatch(r"le[- ]*'?s[- ]?discuss(?:\s+usd)?", lower):
        return "Let's discuss"
    if re.fullmatch(r"neg[- ]?ia[- ]?i[- ]?n(?:\s+usd)?", lower):
        return 'Negotiable'
    if re.search(r"y[- ]*['\-]?u[- ]*['\-]?ll", lower) and 'ove' in lower:
        return "You'll love it"
    return ''


NEGOTIATION_LABELS = {
    "You'll love it", "Attractive", "Competitive",
    "Negotiable", "Let's discuss", "Open to negotiation",
    "Negotiation", "Thương lượng"
}

def normalize_text_salary(text: str) -> str:
    """Chuẩn hóa dấu câu, viết tắt đơn vị."""
    text = text.strip()
    text = text.replace('−', '-').replace('–', '-').replace('—', '-')
    text = re.sub(r"['']", "'", text)
    text = re.sub(r'[""]', '"', text)
    text = re.sub(r'\s*-\s*', '-', text)
    text = re.sub(r'(?i)\bup[- ]*to\b', 'Up to', text)
    text = re.sub(r'(?i)\bgr[- ]?ss\b', '', text)           # gross → xóa
    text = re.sub(r'(?i)\bne[- ]?usd\b', 'USD', text)
    text = re.sub(r'(?i)\bmil\b', 'm', text)
    text = re.sub(r'(?i)\btri[eệ]u\b', 'm', text)
    text = re.sub(r'(?i)\btr\b', 'm', text)
    text = re.sub(r'(?<=\d)-?r\b', 'm', text)               # 20-r → 20m
    # VNW hay dùng "triệu đồng/tháng" → đã xử lý qua 'tr'→'m'
    text = re.sub(r'(?i)\bđồng\b', '', text)
    text = re.sub(r'(?i)/\s*tháng\b', '', text)             # bỏ "/tháng" sau khi đã chuẩn hóa
    text = re.sub(r'\s+', ' ', text).strip()
    special = normalize_special_phrase(text)
    if special:
        return special
    return text


# --- 5.2. Trích xuất số ---

def normalize_number_token(token: str):
    token = re.sub(r'[^\d.,]', '', token.replace(' ', ''))
    if not token:
        return None
    if not re.search(r'[.,]', token):
        return float(token)
    parts = [p for p in re.split(r'[.,]', token) if p != '']
    if len(parts) > 1 and all(len(p) == 3 for p in parts[1:]):
        return float(''.join(parts))
    if len(parts) == 2:
        left, right = parts
        if len(right) in (1, 2):
            return float(f'{left}.{right}')
        return float(left + right)
    return float(''.join(parts))


def extract_numbers_from_salary(text: str):
    candidates = re.findall(r'\d+(?:[.,]\d+)?', text)
    nums = []
    for c in candidates:
        v = normalize_number_token(c)
        if v is not None:
            nums.append(v)
    return nums


# --- 5.3. Suy luận tiền tệ ---

def infer_currency(text: str, numbers: list) -> str:
    upper = text.upper()
    if any(kw in upper for kw in ['VND', 'VNĐ', 'DONG', '₫']) or 'Đ' in text:
        return 'VND'
    if '$' in text and 'USD' not in upper:
        return 'USD'
    if 'USD' in upper:
        if any(n > 100_000 for n in numbers): return 'VND'
        if any(n < 100 for n in numbers): return 'VND'
        if any(500 <= n <= 9_999 for n in numbers): return 'USD'
        return 'USD'
    if any(n > 100_000 for n in numbers): return 'VND'
    if any(n < 100 for n in numbers): return 'VND'
    if any(500 <= n <= 9_999 for n in numbers): return 'USD'
    return 'VND'


def format_m(value: float) -> str:
    if abs(value - round(value)) < 1e-9:
        return f'{int(round(value))}m'
    return f'{value:.2f}m'.rstrip('0').rstrip('.')


# --- 5.4. Main salary processor ---

USD_TO_VND   = 25_400
ANNUAL_MONTHS = 12
MIN_VALID_VND = 1_000_000
MAX_VALID_VND = 200_000_000


def process_salary(raw_value) -> dict:
    """
    Làm sạch và parse salary.
    Trả về dict:
        salary_clean  : str  (vd "12m-20m VND", "1500-2000 USD", "Negotiable", "")
        salary_min_vnd: float hoặc NaN
        salary_max_vnd: float hoặc NaN
        salary_avg_vnd: float hoặc NaN
    """
    nan_result = dict(salary_clean='', salary_min_vnd=np.nan,
                      salary_max_vnd=np.nan, salary_avg_vnd=np.nan)

    if pd.isna(raw_value):
        return nan_result

    text = str(raw_value).strip()
    if not text or text.lower() in {'nan', 'none', ''}:
        return nan_result

    # Bước 1: normalize unit lỗi
    text = normalize_salary_unit(text)
    # Bước 2: normalize text
    text = normalize_text_salary(text)

    # Bước 3: negotiation labels
    if text in NEGOTIATION_LABELS:
        return dict(salary_clean=text, salary_min_vnd=np.nan,
                    salary_max_vnd=np.nan, salary_avg_vnd=np.nan)

    # Bước 4: xử lý đơn vị K
    text_for_parse = re.sub(
        r'(\d+(?:\.\d+)?)\s*[kK]\b',
        lambda m: str(float(m.group(1)) * 1000),
        text
    )

    # Bước 5: trích số
    numbers = extract_numbers_from_salary(text_for_parse)
    if not numbers:
        return nan_result

    # Bước 6: currency & annual
    currency = infer_currency(text, numbers)
    is_annual = bool(re.search(r'\bnăm\b|\byear\b|\bannual\b', text.lower()))
    is_upto   = text.lower().startswith('up')

    # Bước 7: xác định min/max
    min_sal, max_sal = np.nan, np.nan
    if len(numbers) >= 2:
        min_sal, max_sal = numbers[0], numbers[1]
    elif len(numbers) == 1:
        if is_upto or any(kw in text.lower() for kw in ['tới', 'upto', 'lên đến', 'max', 'up to']):
            max_sal = numbers[0]
        elif any(kw in text.lower() for kw in ['từ', 'trên', 'min', '>']):
            min_sal = numbers[0]
        else:
            min_sal = max_sal = numbers[0]

    # Bước 8: quy đổi VND
    def to_vnd(n, currency, is_annual):
        if pd.isna(n): return np.nan
        if currency == 'USD': n *= USD_TO_VND
        if is_annual: n /= ANNUAL_MONTHS
        return n

    min_vnd = to_vnd(min_sal, currency, is_annual)
    max_vnd = to_vnd(max_sal, currency, is_annual)

    # Bước 9: kiểm tra khoảng hợp lệ
    valid_vals = [v for v in [min_vnd, max_vnd] if not np.isnan(v)]
    avg_vnd = float(np.mean(valid_vals)) if valid_vals else np.nan
    if np.isnan(avg_vnd) or avg_vnd < MIN_VALID_VND or avg_vnd > MAX_VALID_VND:
        return nan_result

    # Bước 10: format string đẹp
    if currency == 'VND':
        parts = [format_m(n / 1_000_000 if n > 100_000 else n) for n in numbers[:2]]
        salary_clean = '-'.join(parts) + ' VND'
    else:
        parts = [str(int(round(n))) if abs(n - round(n)) < 1e-9 else str(n) for n in numbers[:2]]
        salary_clean = '-'.join(parts) + ' USD'

    if is_upto:
        salary_clean = 'Up to ' + salary_clean

    return dict(salary_clean=salary_clean,
                salary_min_vnd=min_vnd,
                salary_max_vnd=max_vnd,
                salary_avg_vnd=avg_vnd)


# Áp dụng — trả về list rồi bung thành DataFrame (hiệu năng cao)
print("[5] Đang parse salary...")
salary_results = df_vnw['salary'].apply(process_salary).tolist()
df_sal_parsed  = pd.DataFrame(salary_results)
df_vnw = pd.concat([df_vnw.reset_index(drop=True), df_sal_parsed], axis=1)

sal_parsed_count     = df_sal_parsed['salary_avg_vnd'].notna().sum()
sal_negot_count      = df_sal_parsed['salary_clean'].isin(NEGOTIATION_LABELS).sum()
print(f"    → Có số liệu VND: {sal_parsed_count} dòng")
print(f"    → Thương lượng:   {sal_negot_count} dòng")
print(f"    → Trống/lỗi:      {len(df_vnw) - sal_parsed_count - sal_negot_count} dòng")

# 6. CHUẨN HÓA LOCATION & IS_REMOTE
def standardize_location(loc_str):
    if pd.isna(loc_str): return 'Khác'
    s = str(loc_str).lower()

    locations = []

    if re.search(r'\b(hồ chí minh|hcm|ho chi minh|saigon|tphcm)\b', s):
        locations.append('Hồ Chí Minh')

    if re.search(r'\b(hà nội|ha noi|hn)\b', s):
        locations.append('Hà Nội')

    if re.search(r'\b(đà nẵng|da nang|dn)\b', s):
        locations.append('Đà Nẵng')

    # Bổ sung bộ từ khóa Remote đầy đủ cho riêng Location
    if re.search(r'\b(remote|từ xa|wfh|work from home|làm việc tại nhà)\b', s):
        locations.append('Remote')

    if len(locations) > 0:
        return ', '.join(locations)

    return 'Khác'

df_vnw['location_clean'] = df_vnw['location'].apply(standardize_location)

# Quét trên Title, Location và Description để xác định is_remote
# Pattern bắt mọi dấu hiệu làm việc từ xa
REMOTE_PATTERN = r'(?i)\b(remote|wfh|work from home|từ xa|làm việc tại nhà)\b'

# Hàm nội bộ kết hợp nội dung các cột để đánh giá một lần
def detect_remote(row):
    # Nối text của 3 cột (xử lý None/NaN bằng chuỗi rỗng)
    text_corpus = ' '.join([
        str(row.get('title', '')),
        str(row.get('location', '')),
        str(row.get('description', ''))
    ])

    # Nếu tìm thấy bất kỳ keyword nào trong corpus → Đánh cờ 1
    if re.search(REMOTE_PATTERN, text_corpus):
        return 1
    return 0

# Áp dụng vectorize theo dòng (axis=1)
df_vnw['is_remote'] = df_vnw.apply(detect_remote, axis=1)

print(f"[6] Phân bổ Location: {df_vnw['location_clean'].value_counts().to_dict()}")
print(f"[6] Tỉ lệ Remote Job: {df_vnw['is_remote'].sum()} jobs detected.")

# 7. LÀM SẠCH DESCRIPTION & SKILLS
def clean_description(text):
    if pd.isna(text): return ""
    text = str(text)
    text = re.sub(r'<[^>]+>', ' ', text)
    for pattern in [
        r'(?i)mô tả công việc', r'(?i)yêu cầu công việc',
        r'(?i)quyền lợi', r'(?i)cách thức ứng tuyển',
        r'(?i)phúc lợi', r'(?i)địa điểm làm việc',
    ]:
        text = re.sub(pattern, '', text)
    text = re.sub(r'http\S+|www\.\S+|\S+@\S+', '', text)
    text = re.sub(r'[\r\n\t\|\*•]', ' ', text)
    text = re.sub(r'(?m)^[\-\*\•\+]\s*|^\d+\.\s*', '', text)
    return re.sub(r'\s+', ' ', text).strip()

df_vnw['description_clean'] = df_vnw['description'].apply(clean_description)

# 1: TỪ ĐIỂN RÁC MỞ RỘNG (Vét cạn danh mục & Tỉnh thành)
STOP_SKILLS_MASTER = {
    # Nhãn chung
    'tất cả danh mục', 'dành cho nhà tuyển dụng', 'urgent', 'xem bản đồ', 'premium', 'khác',
    'trang văn hóa công ty', 'tập đoàn', 'chi nhánh',
    # Các lĩnh vực IT / Kỹ thuật bị gắn nhãn sai
    'hệ thống cntt & thiết bị', 'hệ thống cntt', 'thiết bị', 'phần cứng máy tính', 'it phần cứng',
    'it phần mềm', 'it support/help desk', 'qa/qc/software testing', 'viễn thông',
    'internet/online media', 'cơ khí', 'ô tô', 'tự động hóa', 'kỹ thuật điện/điện tử',
    'điện/điện tử', 'khoa học & kỹ thuật', 'nghiên cứu & phát triển', 'quản lý dự án công nghệ',
    # Các lĩnh vực Non-IT
    'bán lẻ/tiêu dùng', 'bán lẻ/bán sỉ', 'hàng tiêu dùng', 'dịch vụ khách hàng', 'hành chính văn phòng',
    'nhân sự/tuyển dụng', 'tài chính/đầu tư', 'bất động sản/cho thuê', 'bất động sản', 'xây dựng',
    'kiến trúc/xây dựng', 'giáo dục/đào tạo', 'giáo dục', 'dược phẩm', 'y tế/chăm sóc sức khỏe',
    'thiết bị y tế', 'bảo hiểm', 'vận tải', 'hậu cần/giao nhận', 'chứng khoán', 'pháp lý',
    'nội thất/gỗ', 'bao bì/in ấn/dán nhãn', 'chính phủ & ngo', 'thu mua', 'biên phiên dịch',
    'phát triển kinh doanh', 'kế toán/kiểm toán', 'kế toán', 'kiểm toán', 'truyền thông/báo chí',
    'tiếp thị/marketing', 'tiếp thị', 'marketing', 'bán hàng/kinh doanh', 'bán hàng kỹ thuật',
    'mới tốt nghiệp', 'thực tập', 'nghệ thuật/thiết kế/giải trí', 'thiết kế đồ họa', 'hành chính', 'nhân sự',
    # Các tỉnh thành thường xuyên lọt vào
    'hà nội', 'hồ chí minh', 'đà nẵng', 'hải phòng', 'bắc ninh', 'bắc giang', 'hà nam', 'hưng yên',
    'vĩnh phúc', 'thái nguyên', 'thanh hóa', 'nghệ an', 'đồng nai', 'bình dương', 'bà rịa - vũng tàu',
    'cần thơ', 'đồng tháp', 'bến tre', 'tây ninh', 'cà mau', 'ninh bình', 'phú thọ', 'hải dương', 'hà tĩnh', 'hn', 'hcm', 'tphcm'
    # 1. Rác giao diện & Boilerplate
    'xem chi tiết', 'xem thêm', 'tuyển dụng', 'tuyển gấp sales admin', 'bộ phận tuyển dụng',
    'cv xin việc', 'mẫu cv', 'thần số học', 'head hunter', 'navigos search',
    'navigos search\'s client', 'khác', 'article',

    # 2. Tên Công ty / Đơn vị tổ chức
    'techcombank', 'shinhan bank vietnam', 'ngân hàng tmcp kiên long', 'mizuho bank, ltd.',
    'tập đoàn vinaseed', 'one mount', 'fpt is', 'fpt software', 'yamaha motor vietnam',
    'heineken vietnam', 'abbott', 'bat vietnam', 'hsbc vietnam', 'seaps vietnam co., ltd',
    'itechwx company limited', 'bluebirds pay llc', 'family medical practice',
    'misa jointstock company', 'tập đoàn gotec land', 'concentrix', 'volio vietnam',
    'fastboy marketing', 'aon vietnam', 'tetra pak vietnam', 'eeo việt nam (classin)',
    'crossian việt nam', 'zeder vietnam', 'k&m holdings', 'nakivo', 'viettel vtt',
    'medicia vietnam', 'smartdev llc', 'sharkninja vietnam', 'geniee vietnam co., ltd.',
    'megaevolution inc.', 'aoede llc', 'kpmg việt nam', 'svp vietnam co., ltd',
    'pwc (vietnam) ltd.', 'lr-tek', 'dsv x schenker', 'soji electronics.', 'nexthop ai',
    'innorix vietnam', 'kortek vina co., ltd', 'neyu', 'audience serv', 'lampart co., ltd',
    'electrolux vietnam ltd.', 'motoniq inc', 'dai-ichi life vietnam', 'the grand ho tram',
    'spartronics vietnam', 'qnap systems, inc.', 'cba (china) pte ltd', 'mainetti (vietnam)',
    'phú hưng life', 'cmc cyber security', 'geek up', 'ccstorages', 'sapp academy',
    'rmit university vietnam',

    # 3. Tên người liên hệ
    'ms. đan', 'ms. hường', 'ms linh', 'ms.trinh trần', 'ms dung', 'ms. phương', 'ms. vân',
    'ms. hiền', 'ms. chuc', 'mr.thắng', 'mr đạt', 'mr. pallee', 'nguyễn đức thành',
    'phan hồng hạnh (ms.)', 'nghi trần', 'tram duong', 'khánh hương', 'giovanni pornillosa',
    'nguyen thi thuy',

    # 4. Nhãn ngành nghề / Lĩnh vực
    'ngân hàng', 'tài chính ngân hàng', 'sản xuất', 'phần mềm máy tính', 'phần cứng',
    'kinh doanh', 'kinh tế', 'chuyển đổi số', 'dệt may/may mặc/giày dép', 'dệt may/da giày',
    'thiết kế', 'thời trang/trang sức', 'tài chính', 'kế toán tổng hợp', 'kế toán tài chính',
    'thương mại điện tử', 'tiếp thị trực tuyến', 'tiếp thị thương mại', 'điện tử viễn thông',
    'nhựa & cao su', 'khai khoáng/dầu khí', 'vật liệu xây dựng', 'hoá chất/hoá sinh',
    'nông/lâm/ngư nghiệp', 'quốc tế',

    # 5. Chức danh công việc & Phòng ban
    'back end developer', 'frontend developer', 'senior data engineer', 'data scientist',
    'ai engineer', '[ho] ai engineer', 'robotics ai engineer', 'database administrator',
    'senior security engineer', 'lead developer', 'brse (.net, c#)', 'brse', 'dba',
    'lập trình viên', 'lập trình viên mes', 'scrum master', 'product owner',
    'senior tester (qa)', 'it manager', 'it helpdesk', 'business analyst',
    'junior business analyst', 'account executive', 'chief accountant', 'sales executive',
    'sales staff', 'sales manager', 'thủ kho', 'kế toán kiêm thủ quỹ', 'hr dept',
    'hr depart', 'phòng nhân sự', 'talent acquisition team', 'talent acquisition dept',
    'viettel customer service', 'viettel it center (vic)',

    # 6. Địa danh
    'lào cai', 'gia lai', 'yên bái', 'quảng nam', 'thừa thiên huế', 'khánh hòa', 'long an',

    # 7. Tính cách / Đặc điểm cá nhân
    'thái độ chuyên nghiệp', 'nhanh nhẹn', 'tỉ mỉ cẩn trọng', 'ham học hỏi',
    'trách nhiệm cao', 'năng động'
}

# 2: TỪ KHÓA ĐẶC TRƯNG CỦA CHỨC DANH / TÊN CÔNG TY
TITLE_COMPANY_KEYWORDS = [
    'nhân viên', 'chuyên viên', 'kỹ sư', 'giám đốc', 'quản lý', 'trưởng phòng',
    'phó phòng', 'trợ lý', 'thư ký', 'thực tập sinh', 'công ty', 'tnhh', 'jsc',
    'co., ltd', 'group', 'corporation'
]

def clean_skills_pro(skills_str):
    if pd.isna(skills_str): return ''

    # Tiền xử lý chuỗi: Xóa ký tự trắng ẩn HTML và chuẩn hóa Unicode
    clean_str = str(skills_str).replace('\xa0', ' ').replace('\u200b', '')
    clean_str = unicodedata.normalize('NFC', clean_str)

    raw_skills = clean_str.split(',')
    cleaned_skills = []

    for s in raw_skills:
        s = s.strip()
        s_lower = s.lower()

        # Bỏ qua nếu độ dài vô lý (kỹ năng IT hiếm khi quá 35 ký tự)
        if not (1 < len(s) <= 35):
            continue

        # [Bộ lọc 1] Khớp chính xác với từ điển Master
        if s_lower in STOP_SKILLS_MASTER:
            continue

        # [Bộ lọc 2] Gạt bỏ nếu chứa cụm từ rác hiển nhiên
        if any(trash in s_lower for trash in ['danh mục', 'nhà tuyển dụng', 'bản đồ']):
            continue

        # [Bộ lọc 3] Heuristic Rule: Gạt bỏ nếu giống Tên công ty hoặc Chức danh
        if any(keyword in s_lower for keyword in TITLE_COMPANY_KEYWORDS):
            continue

        cleaned_skills.append(s)

    final_skills = list(dict.fromkeys(cleaned_skills))
    return ', '.join(final_skills)

df_vnw['skills_clean'] = df_vnw['skills'].apply(clean_skills_pro)

df_vnw['skill_count'] = df_vnw['skills_clean'].apply(
    lambda x: len(x.split(',')) if x else 0
)

# 8. XUẤT FILE
df_vnw['source'] = 'VietnamWorks'

final_cols = [
    'source', 'title_clean', 'company',
    'salary',           # raw salary gốc
    'salary_clean',     # salary đã format (vd "12m-20m VND")
    'salary_min_vnd', 'salary_max_vnd', 'salary_avg_vnd',
    'location_clean', 'is_remote',
    'job_level', 'yoe_extracted',
    'skills_clean', 'skill_count',
    'description_clean', 'url'
]

final_cols    = [c for c in final_cols if c in df_vnw.columns]
df_vnw_final  = df_vnw[final_cols]

output_path = 'VIETNAMWORKS_CLEANED_READY_FINAL.csv'
df_vnw_final.to_csv(output_path, index=False, encoding='utf-8-sig')

print(f"[XONG] File lưu tại: {output_path}")
print(f"       Tổng dòng: {len(df_vnw_final)}")
print(f"       Cột: {list(df_vnw_final.columns)}")


 PHASE 1 (VietnamWorks): LÀM SẠCH & CHUẨN HÓA SCHEMA
[1] Header tìm thấy ở dòng 1
[1] Đọc file: 1186 dòng | Cột: ['title', 'company', 'salary', 'location', 'description', 'skills', 'url', 'crawl_time']
[2] Duplicate: 1186 → 1184 dòng (loại 2)
[3] Job level: {'Undefined': 617, 'Junior': 198, 'Senior': 168, 'Manager': 142, 'Lead': 41, 'Fresher': 18}
[5] Đang parse salary...
    → Có số liệu VND: 151 dòng
    → Thương lượng:   714 dòng
    → Trống/lỗi:      319 dòng
[6] Phân bổ Location: {'Hà Nội': 635, 'Hồ Chí Minh': 536, 'Đà Nẵng': 13}
[6] Tỉ lệ Remote Job: 16 jobs detected.

[XONG] File lưu tại: VIETNAMWORKS_CLEANED_READY_FINAL.csv
       Tổng dòng: 1184
       Cột: ['source', 'title_clean', 'company', 'salary', 'salary_clean', 'salary_min_vnd', 'salary_max_vnd', 'salary_avg_vnd', 'location_clean', 'is_remote', 'job_level', 'yoe_extracted', 'skills_clean', 'skill_count', 'description_clean', 'url']


Cell: Xử lý dữ liệu của TopDev

In [ ]:
import re
import unicodedata
import pandas as pd
import numpy as np
from pathlib import Path

# CONFIG
INPUT_FILE  = Path("topdev_autosave.csv")
OUTPUT_FILE = Path("topdev_autosave_clean.csv")
VND_TO_USD  = 25_500
# 1. Kiểm tra dữ liệu TRƯỚC khi sàn lọc
df = pd.read_csv(INPUT_FILE, encoding="utf-8")

print("=== THÔNG TIN DỮ LIỆU GỐC ===")
print(f"Số lượng bản ghi: {df.shape[0]}")
print(f"Số lượng cột: {df.shape[1]}")

print("\n=== THÔNG TIN CÁC CỘT ===")
df.info()

print("\n=== TÌNH TRẠNG NULL ===")
null_raw = df.isnull().sum()
print(null_raw[null_raw > 0])

print("\n=== MẪU 3 DÒNG ĐẦU TIÊN ===")
display(df.head(3))
# 2. Khai báo các hàm xử lý
def clean_title(title: str) -> str:
    if not isinstance(title, str) or title.strip() == "":
        return np.nan
    t = unicodedata.normalize("NFC", title).strip()
    
    # Xóa nội dung trong [] và () (kể cả dấu ngoặc)
    t = re.sub(r"\[.*?\]", "", t)
    t = re.sub(r"\(.*?\)", "", t)
    
    # Xóa từ dấu "-" về sau nếu title có chứa "chuyên viên"
    if re.search(r"chuyên\s*viên", t, re.IGNORECASE):
        t = re.sub(r"\s*-.*", "", t)
        
    t = re.sub(r"[&#+@*=|\\<>{}~`^$!?\"';:]", " ", t)
    t = re.sub(r"\s*/\s*", "/", t)
    t = re.sub(r"^/+|/+$", "", t)
    t = re.sub(r"\s{2,}", " ", t).strip()
    t = re.sub(r"^[,.\-/\s]+|[,.\-/\s]+$", "", t).strip()
    return t if t else np.nan

JOB_LEVEL_MAP = {
    "junior":    "Junior",
    "middle":    "Mid-level",
    "senior":    "Senior",
    "fresher":   "Junior/Fresher",
    "architect": "Lead",
}

def normalize_job_level(raw) -> str:
    if not isinstance(raw, str) or raw.strip() == "":
        return "Undefined"
    return JOB_LEVEL_MAP.get(raw.strip().lower(), raw.strip())

LOCATION_MAP = {
    r"hà\s*nội|ha\s*noi|hanoi":                                  "Hà Nội",
    r"hồ\s*chí\s*minh|ho\s*chi\s*minh|hcm|tp\.?\s*hcm|tphcm":   "Hồ Chí Minh",
    r"đà\s*nẵng|da\s*nang":                                      "Đà Nẵng",
    r"remote|từ xa|work from home|wfh|oversea|nước ngoài":       "Remote",
}

def normalize_location(loc: str) -> str:
    if not isinstance(loc, str):
        return "Khác"
    l = unicodedata.normalize("NFC", loc)
    for pattern, city in LOCATION_MAP.items():
        if re.search(pattern, l, re.IGNORECASE):
            return city
    return "Khác"

def extract_is_remote(loc: str) -> int:
    if not isinstance(loc, str):
        return 0
    keywords = r"remote|từ xa|work from home|wfh|oversea|nước ngoài"
    return 1 if re.search(keywords, loc, re.IGNORECASE) else 0

def clean_description(text: str) -> str:
    if not isinstance(text, str) or text.strip() == "":
        return np.nan
    t = unicodedata.normalize("NFC", text)
    main_headers = r"(?:MÔ TẢ CÔNG VIỆC\s*:?|YÊU CẦU CÔNG VIỆC\s*:?|Your role\s*&\s*responsibilities|Your skills\s*&\s*qualifications)"
    t = re.sub(rf"^.*?{main_headers}", " ", t, flags=re.IGNORECASE|re.DOTALL)
    sub_headers = [
        r"Your skills\s*&\s*qualifications",
        r"Benefits for you",
        r"QUYỀN LỢI\s*:?",
        r"MÔ TẢ CÔNG VIỆC\s*:?",
        r"YÊU CẦU CÔNG VIỆC\s*:?",
        r"Department\s*:",
        r"Report to\s*:",
        r"Số lượng\s*:",
        r"About\s+\w[\w\s]+(?=\n|Your role|MÔ TẢ)", 
    ]
    for pat in sub_headers:
        t = re.sub(pat, " ", t, flags=re.IGNORECASE)
    t = re.sub(r"(?<!\w)\s*-\s+", ". ", t)
    t = re.sub(r"<[^>]+>", " ", t)
    t = re.sub(r"\s{2,}", " ", t).strip()
    t = re.sub(r"^\.\s*", "", t).strip()
    return t if t else np.nan

def clean_skills(skills_raw) -> str:
    if not isinstance(skills_raw, str) or skills_raw.strip() == "":
        return np.nan
    parts = [s.strip() for s in re.split(r"[,;|]", skills_raw) if s.strip()]
    noise_skills = {"ngân hàng", "tài chính", "korean"}
    seen, unique = set(), []
    for p in parts:
        key = p.lower()
        if key in noise_skills:
            continue
        if key not in seen:
            seen.add(key)
            unique.append(p)
    return ", ".join(unique) if unique else np.nan

def count_skills(skills_clean) -> int:
    if not isinstance(skills_clean, str) or skills_clean.strip() == "":
        return 0
    return len([s for s in skills_clean.split(",") if s.strip()])

def parse_yoe(exp_raw) -> float:
    if not isinstance(exp_raw, str):
        return np.nan
    s = exp_raw.strip().lower()
    if s == "unknown":
        return 0.0
    if s in ("", "nan", "none"):
        return np.nan
    nums = re.findall(r"\d+(?:\.\d+)?", s)
    if not nums:
        return np.nan
    values = [float(n) for n in nums]
    yoe_min = min(values)
    if 0 <= yoe_min <= 15:
        return yoe_min
    return np.nan

def _parse_number(s: str) -> float:
    cleaned = s.replace(".", "").replace(",", "")
    return float(cleaned)

def _format_usd(value: float) -> str:
    return f"{int(round(value)):,} USD"

def parse_salary(salary_raw: str):
    if not isinstance(salary_raw, str):
        return (np.nan, np.nan, np.nan)
    s = unicodedata.normalize("NFC", salary_raw).strip()
    if re.search(r"negotiable|thỏa thuận|thoa thuan|thương lượng|thuong luong", s, re.IGNORECASE):
        return ("Negotiable", np.nan, np.nan)
    num_pat = r"[\d]+(?:\.[\d]{3})*(?:,[\d]+)?"
    range_pat = rf"({num_pat})\s*(VND|USD)\s+to\s+({num_pat})\s*(VND|USD)"
    m = re.search(range_pat, s, re.IGNORECASE)
    if m:
        lo_str, lo_cur, hi_str, hi_cur = m.groups()
        lo = _parse_number(lo_str)
        hi = _parse_number(hi_str)
        lo_usd = lo / VND_TO_USD if lo_cur.upper() == "VND" else lo
        hi_usd = hi / VND_TO_USD if hi_cur.upper() == "VND" else hi
        clean = f"{_format_usd(lo_usd)} - {_format_usd(hi_usd)}"
        return (clean, round(lo_usd, 2), round(hi_usd, 2))
    upto_pat = rf"up\s+to\s+({num_pat})\s*(VND|USD)"
    m = re.search(upto_pat, s, re.IGNORECASE)
    if m:
        val_str, cur = m.groups()
        val = _parse_number(val_str)
        val_usd = val / VND_TO_USD if cur.upper() == "VND" else val
        clean = f"Up to {_format_usd(val_usd)}"
        return (clean, np.nan, round(val_usd, 2))
    nums = re.findall(num_pat, s)
    if nums:
        try:
            val = _parse_number(nums[0])
            cur = "USD" if re.search(r"usd|\$", s, re.IGNORECASE) else "VND"
            val_usd = val if cur == "USD" else val / VND_TO_USD
            return (f"{_format_usd(val_usd)}", np.nan, round(val_usd, 2))
        except Exception:
            pass
    return (s, np.nan, np.nan)

def impute_salary(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df["salary_avg_usd"] = df[["salary_min_usd", "salary_max_usd"]].mean(axis=1)
    mask_upto = df["salary_min_usd"].isna() & df["salary_max_usd"].notna()
    df.loc[mask_upto, "salary_avg_usd"] = df.loc[mask_upto, "salary_max_usd"]
    df["salary_final_usd"] = np.nan
    df["impute_method"]    = "actual"
    df["is_AI_predicted"]  = 0

    mask_actual = df["salary_avg_usd"].notna()
    df.loc[mask_actual, "salary_final_usd"] = df.loc[mask_actual, "salary_avg_usd"]

    group_med = df.loc[mask_actual].groupby(["job_level", "location_clean"])["salary_avg_usd"].median()
    for idx in df.index[~mask_actual]:
        key = (df.at[idx, "job_level"], df.at[idx, "location_clean"])
        if key in group_med.index and not pd.isna(group_med[key]):
            df.at[idx, "salary_final_usd"] = group_med[key]
            df.at[idx, "impute_method"]    = "group_median"
            df.at[idx, "is_AI_predicted"]  = 1

    level_med = df.loc[mask_actual].groupby("job_level")["salary_avg_usd"].median()
    still = df["salary_final_usd"].isna()
    for idx in df.index[still]:
        lvl = df.at[idx, "job_level"]
        if lvl in level_med.index and not pd.isna(level_med[lvl]):
            df.at[idx, "salary_final_usd"] = level_med[lvl]
            df.at[idx, "impute_method"]    = "level_median"
            df.at[idx, "is_AI_predicted"]  = 1

    global_med = df.loc[mask_actual, "salary_avg_usd"].median()
    still = df["salary_final_usd"].isna()
    df.loc[still, "salary_final_usd"] = global_med
    df.loc[still, "impute_method"]    = "global_median"
    df.loc[still, "is_AI_predicted"]  = 1

    return df
# 3. Chạy quá trình sàn lọc dữ liệu (Pipeline)
# Đảm bảo không bị trùng
df = df.drop_duplicates(subset=["title", "company", "location"], keep="first")

# Source và loại bỏ cột thừa
df["source"] = "TopDev"
df = df.drop(columns=["crawl_time"], errors="ignore")

# Normalize Unicode
for col in ["title", "company", "salary", "location", "description", "skills", "experience", "job_level", "url"]:
    if col in df.columns:
        df[col] = df[col].apply(lambda x: unicodedata.normalize("NFC", x).strip() if isinstance(x, str) else x)

# Áp dụng các hàm
df["title_clean"] = df["title"].apply(clean_title)
df["job_level"] = df["job_level"].apply(normalize_job_level)
df["location_clean"] = df["location"].apply(normalize_location)
df["is_remote"] = df["location"].apply(extract_is_remote)
df["description_clean"] = df["description"].apply(clean_description)

# Loại bỏ dòng không có kỹ năng từ đầu
df = df.dropna(subset=["skills"])

df["skills_clean"] = df["skills"].apply(clean_skills)

# Loại bỏ dòng không còn kỹ năng sau khi làm sạch (vd: chỉ chứa kỹ năng nhiễu)
df = df.dropna(subset=["skills_clean"])

df["skill_count"] = df["skills_clean"].apply(count_skills)
df["yoe_extracted"] = df["experience"].apply(parse_yoe)

salary_parsed = df["salary"].apply(parse_salary)
df["salary_clean"] = salary_parsed.apply(lambda x: x[0])
df["salary_min_usd"] = pd.to_numeric(salary_parsed.apply(lambda x: x[1]), errors="coerce")
df["salary_max_usd"] = pd.to_numeric(salary_parsed.apply(lambda x: x[2]), errors="coerce")

df_clean = impute_salary(df)

# Sắp xếp lại cột
SCHEMA = [
    "source", "title", "title_clean", "company", "job_level",
    "location", "location_clean", "is_remote",
    "salary", "salary_clean", "salary_min_usd", "salary_max_usd", "salary_avg_usd",
    "salary_final_usd", "impute_method", "is_AI_predicted",
    "description", "description_clean",
    "skills", "skills_clean", "skill_count",
    "experience", "yoe_extracted", "url",
]
ordered = [c for c in SCHEMA if c in df_clean.columns]
rest = [c for c in df_clean.columns if c not in ordered]
df_clean = df_clean[ordered + rest]

print(" Làm sạch xong!")
display(df_clean.head(3))
# 4. Thống kê Null từng dòng/cột và tỉ lệ Null cuối cùng
print("=== THỐNG KÊ NULL TỪNG CỘT SAU KHI CLEAN ===")

null_counts = df_clean.isnull().sum()
null_ratios = (df_clean.isnull().mean() * 100).round(2)

null_summary = pd.DataFrame({
    'Số lượng Null': null_counts,
    'Tỉ lệ Null (%)': null_ratios
})

# Lọc ra các cột có null để dễ nhìn (bạn có thể bỏ lọc để xem toàn bộ)
display(null_summary)

# --- Lưu file ---
df_clean.to_csv(OUTPUT_FILE, index=False, encoding="utf-8-sig")
print(f"\n Lưu dữ liệu làm sạch ra: {OUTPUT_FILE}")



Cell 3: Nối ba file của ba nguồn với nhau

In [ ]:
import pandas as pd
import numpy as np

print(" PHASE 2: HỢP NHẤT & CHUẨN HÓA MASTER DATA (3 NGUỒN)")

# 1. ĐỌC VÀ HỢP NHẤT BA NGUỒN (ITviec, VietnamWorks, TopDev)
# Đọc 2 file đã xử lý từ Phase 1
df_it  = pd.read_csv('ITVIEC_CLEANED_READY_FINAL.csv')
df_vnw = pd.read_csv('VIETNAMWORKS_CLEANED_READY_FINAL.csv')

# Đọc file TopDev từ Google Drive
topdev_path = '/content/drive/MyDrive/ProjectBigData/topdev_autosave_clean_done.csv'
df_topdev = pd.read_csv(topdev_path)

print(f"[1] ITviec:       {len(df_it):>5} dòng | Cột: {len(df_it.columns)}")
print(f"    VietnamWorks: {len(df_vnw):>5} dòng | Cột: {len(df_vnw.columns)}")
print(f"    TopDev:       {len(df_topdev):>5} dòng | Cột: {len(df_topdev.columns)}")

# Đảm bảo ba file có cùng schema trước khi concat
# (Lấy tập hợp duy nhất tất cả các tên cột từ cả 3 dataframe)
all_cols = list(dict.fromkeys(list(df_it.columns) + list(df_vnw.columns) + list(df_topdev.columns)))

df_it     = df_it.reindex(columns=all_cols)
df_vnw    = df_vnw.reindex(columns=all_cols)
df_topdev = df_topdev.reindex(columns=all_cols)

df_full = pd.concat([df_it, df_vnw, df_topdev], ignore_index=True)
print(f"\n[1] Sau concat: {len(df_full)} dòng tổng cộng")

# 2. KHỬ TRÙNG LẶP LIÊN NGUỒN
initial_count = len(df_full)
# Dùng 3 khóa: title_clean + company + location_clean
# → tránh loại nhầm cùng công ty tuyển nhiều vị trí giống tên ở khác địa điểm
DEDUP_KEYS = ['title_clean', 'company', 'location_clean']
dedup_keys_available = [k for k in DEDUP_KEYS if k in df_full.columns]

df_full = (
    df_full
    .drop_duplicates(subset=dedup_keys_available, keep='first')
    .reset_index(drop=True)
)
removed = initial_count - len(df_full)

print(f"\n[2] Khử trùng lặp liên nguồn (key: {dedup_keys_available})")
print(f"    {initial_count} → {len(df_full)} dòng (loại {removed})")
print(f"    Phân phối nguồn:\n{df_full['source'].value_counts().to_string()}")

# 3. ĐIỀN KHUYẾT YOE (Imputation theo nhóm job_level)
# Chiến lược 3 tầng:
#   Tầng 1: Median trong cùng nhóm job_level (sát thực tế nhất)
#   Tầng 2: Median toàn bộ dataset (nếu nhóm đó không có dữ liệu)
#   Tầng 3: KHÔNG dùng 0 — 0 sai về nghĩa (Senior với 0 năm KN là vô lý)

yoe_before_nan = df_full['yoe_extracted'].isna().sum()

# Tầng 1: fillna bằng median của từng nhóm job_level
df_full['yoe_extracted'] = (
    df_full
    .groupby('job_level', group_keys=False)['yoe_extracted']
    .apply(lambda x: x.fillna(x.median()))
)

# Tầng 2: nếu cả nhóm không có YoE nào → dùng median toàn dataset
global_yoe_median = df_full['yoe_extracted'].median()
df_full['yoe_extracted'] = df_full['yoe_extracted'].fillna(global_yoe_median)

yoe_after_nan = df_full['yoe_extracted'].isna().sum()
print(f"\n[3] Impute YoE:")
print(f"    NaN trước: {yoe_before_nan} | NaN sau: {yoe_after_nan}")
print(f"    Median toàn dataset dùng làm fallback: {global_yoe_median:.1f} năm")
print(f"    Median theo nhóm:")
for lvl, med in df_full.groupby('job_level')['yoe_extracted'].median().items():
    print(f"      {lvl:<12}: {med:.1f} năm")

# 4. KIỂM TRA CHẤT LƯỢNG NHANH TRƯỚC KHI XUẤT
print(f"\n[4] Kiểm tra chất lượng:")
# Tỉ lệ có lương
if 'salary_avg_vnd' in df_full.columns:
    sal_rate = df_full['salary_avg_vnd'].notna().mean() * 100
    print(f"    Có salary_avg_vnd: {sal_rate:.1f}%")

# Phân phối job_level
print(f"    Job level:\n{df_full['job_level'].value_counts().to_string()}")

# Salary trung bình theo level
if 'salary_avg_vnd' in df_full.columns:
    print("    Lương trung bình theo level (triệu VND):")
    sal_by_level = (
        df_full.groupby('job_level')['salary_avg_vnd']
        .mean()
        .dropna()
        .sort_values(ascending=False)
    )
    for lvl, avg in sal_by_level.items():
        print(f"      {lvl:<12}: {avg / 1_000_000:.1f}m")

# Cột nào còn NaN nhiều
null_summary = df_full.isnull().mean().sort_values(ascending=False)
high_null = null_summary[null_summary > 0.3]
if not high_null.empty:
    print(f"    [CẢNH BÁO] Cột có >30% giá trị null:")
    for col, rate in high_null.items():
        print(f"      {col:<15}: {rate*100:.1f}%")

# 5. XUẤT FILE
output_filename = 'MASTER_DATA_PRE_IMPUTATION.csv'
df_full.to_csv(output_filename, index=False, encoding='utf-8-sig')

print(f"[HOÀN TẤT] File: {output_filename}")
print(f"           Tổng: {len(df_full)} dòng × {len(df_full.columns)} cột")


 PHASE 2: HỢP NHẤT & CHUẨN HÓA MASTER DATA (3 NGUỒN)
[1] ITviec:         762 dòng | Cột: 16
    VietnamWorks:  1184 dòng | Cột: 16
    TopDev:         217 dòng | Cột: 24

[1] Sau concat: 2163 dòng tổng cộng

[2] Khử trùng lặp liên nguồn (key: ['title_clean', 'company', 'location_clean'])
    2163 → 2064 dòng (loại 99)
    Phân phối nguồn:
source
VietnamWorks    1167
ITviec           761
TopDev           136

[3] Impute YoE:
    NaN trước: 1334 | NaN sau: 0
    Median toàn dataset dùng làm fallback: 3.0 năm
    Median theo nhóm:
      Fresher     : 4.0 năm
      Junior      : 1.0 năm
      Junior/Fresher: 1.0 năm
      Lead        : 5.5 năm
      Manager     : 5.0 năm
      Mid-level   : 3.0 năm
      Senior      : 5.0 năm
      Undefined   : 3.0 năm

[4] Kiểm tra chất lượng:
    Có salary_avg_vnd: 18.0%
    Job level:
job_level
Undefined         1010
Senior             409
Junior             274
Manager            205
Lead                95
Mid-level           29
Fresher             26

Cell 4: Impute salaray

In [ ]:
"""
BƯỚC CUỐI: IMPUTE LƯƠNG & XUẤT FILE (CHIẾN LƯỢC HYBRID)
=========================================================
Chiến lược impute 3 tầng:
  Tầng 1 — Median nhóm (job_level × location_clean)  → ưu tiên
  Tầng 2 — Median nhóm (job_level)                   → fallback nếu tầng 1 thiếu data
  Tầng 3 — RandomForest                               → fallback cuối cùng

Lý do dùng hybrid:
  - Chỉ 16.5% dữ liệu có lương thật → RF train trên sample quá nhỏ (R²=0.18)
  - Median nhóm ổn định hơn nhiều khi data ít
  - RF vẫn hữu ích khi nhóm median không có đủ mẫu

Input : MASTER_DATA_PRE_IMPUTATION.csv
Output: MASTER_DATA_FINAL.csv
        Final_Jobs_Dataset_For_MongoDB.json
        Final_Jobs_Dataset_For_HDFS.tsv
"""

import pandas as pd
import numpy as np
import scipy.sparse as sp
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.metrics import mean_absolute_error, r2_score

print(" BƯỚC CUỐI: IMPUTE LƯƠNG (HYBRID) & XUẤT FILE")

# 1. ĐỌC DỮ LIỆU
df = pd.read_csv('MASTER_DATA_PRE_IMPUTATION.csv')
print(f"[1] Đọc file: {len(df)} dòng × {len(df.columns)} cột")

MIN_VALID_VND = 1_000_000
MAX_VALID_VND = 200_000_000

has_salary  = df['salary_avg_vnd'].notna().values
miss_salary = ~has_salary
print(f"    Có lương thật   : {has_salary.sum()} ({has_salary.sum()/len(df)*100:.1f}%)")
print(f"    Thiếu lương     : {miss_salary.sum()} ({miss_salary.sum()/len(df)*100:.1f}%)")

# 2. TẦNG 1 & 2: MEDIAN THEO NHÓM
print("\n[2] Tầng 1&2: Impute bằng Median nhóm...")

df['salary_final_vnd'] = df['salary_avg_vnd'].copy()
df['impute_method']    = 'actual'  # nhãn theo dõi phương pháp

# --- Tầng 1: median(job_level × location_clean) ---
group1 = df[has_salary].groupby(
    ['job_level', 'location_clean']
)['salary_avg_vnd'].median()

# --- Tầng 2: median(job_level) ---
group2 = df[has_salary].groupby('job_level')['salary_avg_vnd'].median()

# --- Tầng 3 fallback cuối: median toàn dataset ---
global_median = df.loc[has_salary, 'salary_avg_vnd'].median()

print(f"    Median theo nhóm (job_level × location_clean):")
for (lvl, loc), med in group1.items():
    print(f"      {lvl:<12} × {loc:<16}: {med/1_000_000:.1f}m VND")

print(f"\n    Median theo job_level:")
for lvl, med in group2.items():
    print(f"      {lvl:<12}: {med/1_000_000:.1f}m VND")

print(f"\n    Global median: {global_median/1_000_000:.1f}m VND")

# Gán median theo từng tầng cho dòng thiếu lương
tier1_count = tier2_count = 0
still_missing = []

for idx in df[miss_salary].index:
    lvl = df.at[idx, 'job_level']
    loc = df.at[idx, 'location_clean']

    # Tầng 1
    if (lvl, loc) in group1.index and not np.isnan(group1[(lvl, loc)]):
        df.at[idx, 'salary_final_vnd'] = group1[(lvl, loc)]
        df.at[idx, 'impute_method']    = 'median_group1'
        tier1_count += 1
    # Tầng 2
    elif lvl in group2.index and not np.isnan(group2[lvl]):
        df.at[idx, 'salary_final_vnd'] = group2[lvl]
        df.at[idx, 'impute_method']    = 'median_group2'
        tier2_count += 1
    else:
        still_missing.append(idx)

print(f"\n    Imputed Tầng 1 (nhóm level×loc) : {tier1_count} dòng")
print(f"    Imputed Tầng 2 (nhóm level)      : {tier2_count} dòng")
print(f"    Còn lại cần RF                   : {len(still_missing)} dòng")

# 3. TẦNG 3: RANDOM FOREST (chỉ cho dòng còn thiếu sau tầng 1&2)
rf_count = 0

if len(still_missing) > 0 or True: 
    print("\n[3] Tầng 3: Train RandomForest & Đánh giá...")

    # Encode features
    le_level = LabelEncoder()
    le_loc   = LabelEncoder()
    df['level_enc'] = le_level.fit_transform(df['job_level'].fillna('Undefined'))
    df['loc_enc']   = le_loc.fit_transform(df['location_clean'].fillna('Khác'))

    tfidf = TfidfVectorizer(max_features=100, min_df=2)
    skills_matrix = tfidf.fit_transform(df['skills_clean'].fillna(''))

    numeric_features = df[['loc_enc', 'level_enc', 'yoe_extracted', 'skill_count']].fillna(0).values
    X_full = sp.hstack((numeric_features, skills_matrix)).tocsr()

    X_labeled = X_full[has_salary]
    y_labeled = df.loc[has_salary, 'salary_avg_vnd'].values

    # --- Đánh giá bằng 5-fold CV ---
    rf_eval = RandomForestRegressor(
        n_estimators=200,
        min_samples_leaf=2,
        max_features='sqrt',
        random_state=42,
        n_jobs=-1
    )

    cv_mae = -cross_val_score(rf_eval, X_labeled, y_labeled,
                               cv=5, scoring='neg_mean_absolute_error', n_jobs=-1)
    cv_r2  = cross_val_score(rf_eval, X_labeled, y_labeled,
                              cv=5, scoring='r2', n_jobs=-1)

    print(f"\n    === 5-FOLD CROSS VALIDATION (RF trên {has_salary.sum()} mẫu) ===")
    print(f"    Lương TB thực tế  : {y_labeled.mean():>15,.0f} VND")
    print(f"    MAE (mean ± std)  : {cv_mae.mean():>15,.0f} ± {cv_mae.std():,.0f} VND")
    print(f"    MAE/Lương TB      : {cv_mae.mean()/y_labeled.mean()*100:>14.1f}%  ← tỉ lệ sai số")
    print(f"    R²  (mean ± std)  : {cv_r2.mean():>15.4f} ± {cv_r2.std():.4f}")

    # Giải thích R²
    r2_avg = cv_r2.mean()
    if r2_avg >= 0.6:
        r2_note = "Tốt "
    elif r2_avg >= 0.35:
        r2_note = "Chấp nhận được  (do sample nhỏ)"
    else:
        r2_note = "Yếu  — kết quả RF chỉ mang tính tham khảo"
    print(f"    →  R²     : {r2_note}")
    print(f"    →  R² thấp vì chỉ có {has_salary.sum()} mẫu có lương ({has_salary.sum()/len(df)*100:.0f}%)")

    # Hold-out test
    X_train, X_test, y_train, y_test = train_test_split(
        X_labeled, y_labeled, test_size=0.2, random_state=42
    )
    rf_eval.fit(X_train, y_train)
    y_pred = rf_eval.predict(X_test)

    print(f"\n    === HOLD-OUT TEST SET (20%) ===")
    print(f"    MAE : {mean_absolute_error(y_test, y_pred):>15,.0f} VND")
    print(f"    R²  : {r2_score(y_test, y_pred):>15.4f}")

    # MAE theo level
    df_test_eval = df[has_salary].iloc[X_train.shape[0]:].copy()
    if len(df_test_eval) == len(y_test):
        df_test_eval['abs_err'] = np.abs(y_test - y_pred)
        print(f"\n    MAE theo job_level:")
        for lvl, grp in df_test_eval.groupby('job_level'):
            print(f"      {lvl:<12}: {grp['abs_err'].mean():>12,.0f} VND  (n={len(grp)})")

    # --- Train final RF & impute dòng còn thiếu ---
    if len(still_missing) > 0:
        rf_final = RandomForestRegressor(
            n_estimators=200, min_samples_leaf=2,
            max_features='sqrt', random_state=42, n_jobs=-1
        )
        rf_final.fit(X_labeled, y_labeled)

        still_mask = df.index.isin(still_missing)
        predicted  = rf_final.predict(X_full[still_mask])
        predicted  = np.clip(predicted, MIN_VALID_VND, MAX_VALID_VND)

        df.loc[still_mask, 'salary_final_vnd'] = predicted
        df.loc[still_mask, 'impute_method']    = 'random_forest'
        rf_count = len(still_missing)
        print(f"\n    RF imputed {rf_count} dòng còn lại")
    else:
        print("\n    Không còn dòng nào cần RF impute ")

# 4. THỐNG KÊ IMPUTATION
print("\n[4] Tổng kết phương pháp impute:")
method_counts = df['impute_method'].value_counts()
for method, count in method_counts.items():
    pct = count / len(df) * 100
    label = {
        'actual':        ' Lương thật',
        'median_group1': ' Median (level×location)',
        'median_group2': ' Median (level)',
        'random_forest': ' RandomForest',
    }.get(method, method)
    print(f"    {label:<35}: {count:>5} ({pct:.1f}%)")

# Clip toàn bộ salary_final_vnd vào khoảng hợp lệ
df['salary_final_vnd'] = df['salary_final_vnd'].clip(MIN_VALID_VND, MAX_VALID_VND)

# is_AI_predicted: 0 = lương thật, 1 = imputed bất kỳ cách nào
df['is_AI_predicted'] = (df['impute_method'] != 'actual').astype(int)

# 5. XUẤT FILE
print("\n[5] Xuất file...")

OUTPUT_COLS = [
    'source', 'title_clean', 'company',
    'salary',
    'salary_clean',
    'salary_min_vnd', 'salary_max_vnd',
    'salary_final_vnd',
    'is_AI_predicted', 'impute_method',
    'location_clean', 'is_remote',
    'job_level', 'yoe_extracted',
    'skills_clean', 'skill_count',
    'description_clean', 'url'
]
final_cols = [c for c in OUTPUT_COLS if c in df.columns]
df_final   = df[final_cols].copy()
df_final   = df_final.drop(columns=['level_enc', 'loc_enc'], errors='ignore')

df_final.to_csv('MASTER_DATA_FINAL.csv', index=False, encoding='utf-8-sig')
print(f"    [CSV]  MASTER_DATA_FINAL.csv                — {len(df_final)} rows")

df_final.to_json('Final_Jobs_Dataset_For_MongoDB.json',
                 orient='records', lines=True, force_ascii=False)
print(f"    [JSON] Final_Jobs_Dataset_For_MongoDB.json  — {len(df_final)} documents")

df_final.to_csv('Final_Jobs_Dataset_For_HDFS.tsv',
                sep='\t', index=False, encoding='utf-8-sig')
print(f"    [TSV]  Final_Jobs_Dataset_For_HDFS.tsv      — {len(df_final)} rows")

# 6. BÁO CÁO CHẤT LƯỢNG CUỐI
print(f" THỐNG KÊ CHẤT LƯỢNG CUỐI")
print(f"  Tổng dòng         : {len(df_final)}")

print(f"\n  Lương TB theo job_level (salary_final_vnd):")
for lvl, grp in df_final.groupby('job_level')['salary_final_vnd']:
    print(f"    {lvl:<12}: {grp.mean()/1_000_000:>6.1f}m VND  (n={len(grp)})")

# Cảnh báo cột thực sự cần quan tâm (bỏ qua salary_clean, min, max vì null là bình thường)
IGNORE_NULL_WARN = {'salary_clean', 'salary_min_vnd', 'salary_max_vnd'}
null_rate = df_final.isnull().mean()
high_null = null_rate[(null_rate > 0.3) & (~null_rate.index.isin(IGNORE_NULL_WARN))]
if not high_null.empty:
    print(f"\n  [CẢNH BÁO] Cột >30% null (cần xem xét):")
    for col, rate in high_null.items():
        print(f"    {col}: {rate*100:.1f}%")
else:
    print(f"\n  Không có cột quan trọng nào >30% null")



 BƯỚC CUỐI: IMPUTE LƯƠNG (HYBRID) & XUẤT FILE
[1] Đọc file: 2064 dòng × 24 cột
    Có lương thật   : 371 (18.0%)
    Thiếu lương     : 1693 (82.0%)

[2] Tầng 1&2: Impute bằng Median nhóm...
    Median theo nhóm (job_level × location_clean):
      Fresher      × Hà Nội          : 14.0m VND
      Fresher      × Hồ Chí Minh     : 11.4m VND
      Junior       × Hà Nội          : 19.1m VND
      Junior       × Hồ Chí Minh     : 19.8m VND
      Junior       × Khác            : 22.0m VND
      Junior       × Đà Nẵng         : 22.0m VND
      Junior/Fresher × Hà Nội          : 28.0m VND
      Junior/Fresher × Hồ Chí Minh     : 15.3m VND
      Junior/Fresher × Remote          : 50.0m VND
      Lead         × Hà Nội          : 45.7m VND
      Lead         × Hồ Chí Minh     : 56.5m VND
      Lead         × Hồ Chí Minh, Hà Nội: 57.1m VND
      Lead         × Remote          : 63.8m VND
      Lead         × Đà Nẵng         : 38.1m VND
      Manager      × Hà Nội          : 59.7m VND
      Manager  

Làm sạch các dấu đầu dòng còn sót


In [ ]:
import pandas as pd
import re
import unicodedata
print("ĐANG DỌN DẸP RÁC UI, BULLETS VÀ KÝ TỰ ĐẶC BIỆT...")
df = pd.read_csv('MASTER_DATA_FINAL.csv')
# 1. HÀM DỌN RÁC CỘT SKILLS
def remove_ui_artifacts(skills_str):
    if pd.isna(skills_str): return ""
    skills = str(skills_str).split(',')
    bad_phrases = ['xem chi tiết', 'xem bản đồ', 'xem thêm', 'chi tiết']
    cleaned = []
    for s in skills:
        s = s.strip()
        if not any(bad in s.lower() for bad in bad_phrases) and len(s) > 0:
            cleaned.append(s)
    return ', '.join(cleaned)
df['skills_clean'] = df['skills_clean'].apply(remove_ui_artifacts)
df['skill_count']  = df['skills_clean'].apply(lambda x: len(x.split(',')) if x != '' else 0)
# 2. HÀM LỌC KÝ TỰ ĐẶC BIỆT / EMOJI / UNICODE LẠ
# Giữ lại: chữ cái Latin, chữ cái tiếng Việt (Unicode), số, dấu câu thông thường,
#          ký tự kỹ thuật thường gặp trong IT (/, \, @, #, _, +, -, =, <, >, .)
# Loại bỏ: emoji, ký hiệu tiền tệ lạ, ký tự hộp vẽ, arrow Unicode, v.v.
# Các Unicode category cần GIỮ LẠI:
#   L  = chữ cái (Letter)  — bao gồm tiếng Việt, Latin
#   N  = số (Number)
#   Z  = khoảng trắng (Separator)
#   P  = dấu câu (Punctuation)
#   S  = ký hiệu (Symbol) — NHƯNG chỉ giữ ký hiệu toán học thông dụng
ALLOWED_CATEGORIES = {'L', 'N', 'Z', 'P'}
# Ký hiệu (category S) được giữ lại thủ công (phổ biến trong IT/tuyển dụng)
ALLOWED_SYMBOLS = set('$€£¥%@#&*+=<>|\\/_~^`°©®™±÷×')
# Ký tự bullet hợp lệ (được xử lý sau bằng bước 3)
ALLOWED_BULLETS = set('-+•●○▪▸►◆')
def remove_special_chars(text: str) -> str:
    """
    Xóa emoji và ký tự Unicode không liên quan.
    Giữ nguyên: tiếng Việt, Latin, số, dấu câu, ký hiệu IT thông dụng.
    """
    if not text:
        return text
    result = []
    for ch in text:
        cat = unicodedata.category(ch)
        cat1 = cat[0]  # Lấy category chính (L, N, Z, P, S, C, M...)
        if cat1 in ALLOWED_CATEGORIES:
            result.append(ch)
        elif ch in ALLOWED_SYMBOLS:
            result.append(ch)
        elif ch in ALLOWED_BULLETS:
            result.append(ch)    # Giữ lại để bước 3 xử lý
        elif cat1 == 'M':        # Mark (dấu kết hợp tiếng Việt)
            result.append(ch)
        elif cat1 == 'S':        # Symbol — chỉ giữ nếu trong whitelist
            pass                 # Emoji, mũi tên lạ, ký hiệu hộp... → bỏ
        else:
            pass                 # C (control), format chars → bỏ
    return ''.join(result)
# ==========================================
# 3. HÀM DỌN SẠCH BULLETS VÀ DẤU LẠ TRONG DESCRIPTION
# ==========================================
def final_clean_description(text):
    if pd.isna(text): return ""
    text = str(text)
    # Bước A: Xóa ký tự đặc biệt / emoji trước
    text = remove_special_chars(text)
    # Bước B: Xóa Headline bị sót (smart quotes)
    for p in [
        r"(?i)Why You[''\u2019\u2018]ll Love Working Here",
        r"(?i)About our client",
        r"(?i)About the company",
    ]:
        text = re.sub(p, ' ', text)
    # Bước C: Xóa bullets đứng chơ vơ — ký tự đứng 1 mình giữa khoảng trắng
    # Lưu ý: "C++" và "e-commerce" KHÔNG bị ảnh hưởng vì không có space bao quanh
    text = re.sub(r'(?<=\s)[+\-•●○▪▸►◆](?=\s)', ' ', text)
    text = re.sub(r'^[+\-•●○▪▸►◆]\s*', ' ', text, flags=re.MULTILINE)
    # Bước D: Xóa đánh số thứ tự đầu dòng "1. ", "2. " ...
    text = re.sub(r'(?<=\s)\d+\.\s', ' ', text)
    text = re.sub(r'^\d+\.\s', ' ', text, flags=re.MULTILINE)
    # Bước E: Chuẩn hóa khoảng trắng thừa
    return re.sub(r'\s+', ' ', text).strip()
# ==========================================
# 4. ÁP DỤNG VÀO DESCRIPTION & SKILLS
# ==========================================
# Description: full pipeline (emoji → headlines → bullets → numbering)
df['description_clean'] = df['description_clean'].apply(final_clean_description)
# Skills: chỉ cần bỏ emoji/ký tự lạ, giữ dấu phẩy và chữ thường
def clean_skills_unicode(skills_str):
    if pd.isna(skills_str): return ""
    # Xóa ký tự lạ nhưng giữ dấu phẩy, gạch ngang, dấu chấm (vd: "Node.js", "C++", "ASP.NET")
    cleaned = remove_special_chars(str(skills_str))
    # Chuẩn hóa khoảng trắng trong từng skill
    skills = [s.strip() for s in cleaned.split(',') if s.strip()]
    return ', '.join(skills)
df['skills_clean'] = df['skills_clean'].apply(clean_skills_unicode)
df['skill_count']  = df['skills_clean'].apply(lambda x: len(x.split(',')) if x != '' else 0)
# ==========================================
# 5. BÁO CÁO & XUẤT FILE
# ==========================================
print(f"  Tổng dòng        : {len(df)}")
print(f"  skill_count TB   : {df['skill_count'].mean():.1f} skills/job")
print(f"  description dài TB: {df['description_clean'].str.len().mean():.0f} ký tự")
# Kiểm tra nhanh: tìm dòng còn chứa emoji (codepoint > U+1F000)
sample_emoji = df['description_clean'].apply(
    lambda x: any(ord(c) > 0x1F000 for c in str(x)) if pd.notna(x) else False
)
emoji_remaining = sample_emoji.sum()
if emoji_remaining == 0:
    print("  Không còn emoji trong description_clean")
else:
    print(f"  Không Còn {emoji_remaining} dòng có thể chứa emoji — kiểm tra thủ công")
output_file = 'Data_ITJOB_Cleaned.csv'
df.to_csv(output_file, index=False, encoding='utf-8-sig')
print(f"\n File '{output_file}' đã được làm sạch.")


ĐANG DỌN DẸP RÁC UI, BULLETS VÀ KÝ TỰ ĐẶC BIỆT...
  Tổng dòng        : 2064
  skill_count TB   : 4.6 skills/job
  description dài TB: 1321 ký tự
  Không còn emoji trong description_clean

 File 'Data_ITJOB_Cleaned.csv' đã được làm sạch.
